# Layer-fraction sweep: Qwen3-32B -> prompts -> activations -> arc lengths

Runs every stage of the arc-length pipeline for a single model, `Qwen/Qwen3-32B`, once per entry in `LAYER_FRACTIONS`. Each fraction caches a different layer, `floor(frac * num_hidden_layers)`, and writes to its own artifact tree under `artifacts/layer_fraction_sweep/frac<NN>/<model slug>/`, so the sweeps never overwrite one another.

Raw model activations are **not** retained: every `.pt` activation cache is deleted as soon as the artifacts derived from it exist. Fitting caches are discarded after the surface is exported and plotted, and each inference cache is discarded after its CSV is written. Generated datasets, surface artifacts, plots, and inference CSVs are kept. Because nothing raw persists, every fraction recomputes its activations from scratch and `FORCE` never has a cache to reuse.

The model is loaded once per fraction and shared by every stage of that fraction, and its Hugging Face download cache is removed once, after the final fraction, rather than after each one.

In [ ]:
from collections import Counter
from pathlib import Path
import gc
import math
import sys
import traceback

import torch
from huggingface_hub import scan_cache_dir
from IPython.display import display
from transformers import AutoConfig

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'scripts' / 'stakes_surface_pipeline.py').is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scripts import (cache_activations, context_inference, prompt_datasets, severity_flipped_inference,
                     severity_inference, severity_pairwise_inference, severity_wording_inference)
from scripts.pipeline_config import NO_TIME_CORPORA, RunConfig, naming_convention_spec
from scripts.stakes_surface_pipeline import StakesSurfacePipeline

print('Repository:', ROOT)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none (CPU)')

## Configuration

`MODEL` is a single `(model_name, naming_convention)` pair. Use `"llama"` for models whose decoder blocks are named `model.layers.N`, and `"gemma4"` for Gemma 4 multimodal checkpoints. To support another convention, add its dotted config layer-count attribute and module-path template to `NAMING_CONVENTIONS` in `scripts/pipeline_config.py`; add a loader branch in `cache_activations.load_model` only if the architecture cannot use the standard causal-LM loader. Cache keys remain `layer_out/N` for downstream compatibility. For gated models, authenticate before starting. `BATCH_SIZE` applies to every fraction; lower it if needed.

`HF_CACHE_DIR` is the directory Hugging Face downloads model weights into. Leave it `None` to keep Hugging Face's own default (`HF_HOME/hub`, or `HF_HUB_CACHE` when that is set); give it a path to put the weights on a larger or faster volume, which matters on hosts whose home directory is small. The path is expanded and made absolute, and the same directory is used for loading and for the cleanup scan at the end, so a custom location is still reclaimed.

`LAYER_FRACTIONS` lists the depth fractions to sweep; each one is resolved to `floor(frac * num_hidden_layers)`. Two fractions that round to the same layer index are rejected before any work starts. `SWEEP_ROOT` is the artifact root the per-fraction run directories hang off.

In [ ]:
MODEL = ('Qwen/Qwen3-32B', 'llama')
LAYER_FRACTIONS = [0.4, 0.5, 0.6, 0.7, 0.8]
SWEEP_ROOT = ROOT / 'artifacts' / 'layer_fraction_sweep'
BATCH_SIZE = 256
STAKES_MERGES = {
    'near_existential': 'existential',
    'medium_low': 'medium',
}
POSITION = -1                    # final prompt token
CORPORA = list(NO_TIME_CORPORA)  # quick smoke run: ['conversational_no_time']
FORCE = False                    # rebuild caches that fail fingerprint checks
HF_CACHE_DIR = None              # where model weights are downloaded; None = Hugging Face default

## Pipeline helpers

`process_fraction` loads the model once per fraction and reuses that instance for the fitting caches and for all five inference datasets, each of which would otherwise load its own copy. The weights therefore stay resident while the surface is fit; fitting and plotting are CPU work, so nothing else competes for them. The model is released before the fraction returns, so only one is ever loaded at a time.

`report_placement` prints the accelerate `device_map="auto"` placement that `cache_activations.load_model` produces. When every shard lands on a GPU, holding the model through the fitting stage costs VRAM that nothing else wants. When accelerate offloads shards to `cpu` or `disk`, those shards share host RAM with the activation tables the fitting stage loads, and the cell warns so the run can be resized before it reaches the fitting stage.

`discard_activation_caches` is the only departure from the reference pipeline's bookkeeping: it deletes `.pt` activation caches (and any interrupted `.pt.tmp` writes) under one directory, then removes the directories left empty. It is scoped to a single run's own activation folders and never touches datasets, surface artifacts, plots, or CSVs.

Cache cleanup is repository-scoped: it removes all cached revisions of the completed model ID while allowing Hugging Face to preserve blobs shared by other repositories. It never targets this repository's `artifacts/` tree.

In [ ]:
def model_layer_count(model_name, naming_convention, hf_cache_dir=None):
    count_attribute, _ = naming_convention_spec(naming_convention)
    architecture = AutoConfig.from_pretrained(model_name, trust_remote_code=True,
                                              cache_dir=None if hf_cache_dir is None else str(hf_cache_dir))
    layer_config = architecture
    for attribute in count_attribute.split('.'):
        layer_config = getattr(layer_config, attribute, None)
        if layer_config is None:
            break
    num_layers = layer_config
    if not isinstance(num_layers, int) or num_layers < 1:
        raise ValueError(f'{model_name}: config has no positive integer {count_attribute}')
    return num_layers


def cached_layer_for_fraction(model_name, num_layers, fraction):
    layer = math.floor(fraction * num_layers)
    if not 0 <= layer < num_layers:
        raise ValueError(f'{model_name}: computed invalid layer {layer} for {num_layers} layers at fraction {fraction}')
    return layer


def fraction_tag(fraction):
    return f'frac{round(fraction * 100):02d}'


def report_placement(model):
    """Summarise an accelerate `device_map='auto'` placement and flag CPU/disk offload.

    The model is held for the whole fraction, so any offloaded shard sits in host RAM
    while the surface is fit from activation tables that live there too.
    """
    placement = getattr(model, 'hf_device_map', None)
    if not placement:
        print(f'Placement: single device, {model.device}')
        return False
    counts = Counter(str(device) for device in placement.values())
    print('Placement:', ', '.join(f'{device}: {n} modules' for device, n in sorted(counts.items())))
    offloaded = sorted(device for device in counts if device in ('cpu', 'disk'))
    if offloaded:
        print(f'WARNING: {offloaded} offload in use. The model stays loaded through the fitting '
              f'stage, so its offloaded shards compete with the activation tables for host RAM. '
              f'Reduce BATCH_SIZE or CORPORA if a fraction runs out of memory.')
    return bool(offloaded)


def discard_activation_caches(directory):
    """Delete raw activation caches under `directory`; keep every derived artifact."""
    directory = Path(directory)
    if not directory.exists():
        return 0
    freed = 0
    for path in sorted(directory.rglob('*.pt')) + sorted(directory.rglob('*.pt.tmp')):
        freed += path.stat().st_size
        path.unlink()
    for path in sorted(directory.rglob('*'), reverse=True) + [directory]:
        if path.is_dir() and not any(path.iterdir()):
            path.rmdir()
    print(f'Discarded {freed / 2**20:.1f} MiB of raw activations from {directory}', flush=True)
    return freed


def delete_local_model_data(model_name, hf_cache_dir=None):
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    if hf_cache_dir is not None and not Path(hf_cache_dir).is_dir():
        print(f'No Hugging Face cache directory at {hf_cache_dir}; nothing to delete.')
        return
    cache = scan_cache_dir(hf_cache_dir)
    revisions = [revision.commit_hash
                 for repo in cache.repos
                 if repo.repo_type == 'model' and repo.repo_id == model_name
                 for revision in repo.revisions]
    if not revisions:
        print(f'No local Hugging Face model cache remains for {model_name}.')
        return
    strategy = cache.delete_revisions(*revisions)
    print(f'Deleting {model_name} download cache ({strategy.expected_freed_size_str}) ...')
    strategy.execute()


def show_inference_result(label, result):
    csv_path, rows, diagnostics = result
    display(rows)
    display(diagnostics.groupby('coordinate_status').size().rename('rows').to_frame())
    display(diagnostics[['outside_saved_height_range', 'surface_extended', 'height_extrapolated']].sum().rename('rows').to_frame())
    display(diagnostics[['surface_projection_residual', 'slice_height_error']].describe())
    print(f'{label} CSV:', csv_path)


def run_inference_stage(config, label, module, dataset_name, model, tokenizer):
    """Project one inference dataset on the loaded model, then discard the activations it cached."""
    try:
        show_inference_result(label, module.run(config, force=FORCE, model=model, tokenizer=tokenizer))
    finally:
        discard_activation_caches(config.run_dir / 'inference' / dataset_name / 'activations')


def process_fraction(model_name, naming_convention, num_layers, fraction):
    layer = cached_layer_for_fraction(model_name, num_layers, fraction)
    config = RunConfig(model_name=model_name, naming_convention=naming_convention,
                       layer_component=f'layer_out/{layer}',
                       position=POSITION, batch_size=BATCH_SIZE, stakes_merges=STAKES_MERGES,
                       artifact_root=SWEEP_ROOT / fraction_tag(fraction),
                       hf_cache_dir=HF_CACHE_DIR)
    print(f'\n=== {model_name}: caching layer floor({fraction} * {num_layers}) = {layer} ===')
    print(config.describe())
    print(f'Cached module: {config.layer_module_name} ({naming_convention})')

    model = tokenizer = None
    try:
        datasets = prompt_datasets.generate_datasets(config, CORPORA)
        prompt_datasets.preflight(datasets)
        model, tokenizer = cache_activations.load_model(config)
        print(f'Loaded {model_name} on {model.device}')
        report_placement(model)
        cache_paths = cache_activations.run(config, datasets, model=model, tokenizer=tokenizer, force=FORCE)
        print(f'{len(cache_paths)} template caches in {config.activations_dir}')

        pipeline = None
        try:
            pipeline = StakesSurfacePipeline(config)
            display(pipeline.load_caches())
            projected_rows = pipeline.fit_bcpc()
            bcpc = pipeline.bcpc
            print(f'{len(projected_rows):,} rows, {len(bcpc.projection["classes"])} merged classes')
            display(bcpc.class_table()); display(bcpc.variance_table())
            display(bcpc.centroids); display(bcpc.anchors)
            print(f'Weighted squared residual sum: {bcpc.weighted_residual_sum:.6g}')
            display(bcpc.spline_points)
            print(f's=0: spline endpoint associated with {bcpc.zero_anchor}')
            print(f'Total spline arc length: {bcpc.total_arc_length:.6g}')
            display(projected_rows[['stakes', 'spline_parameter', 'bcpc_arc_length', 'distance_to_spline']].head())
            display(pipeline.fit_pls()); display(pipeline.weight_summary)
            rotation_diagnostics, file_centroids = pipeline.rotate_plane()
            display(rotation_diagnostics); display(file_centroids)
            display(pipeline.fit_surface())
            display(pipeline.project_stated_rows())
            display(pipeline.build_slice_cache())
            mapping = pipeline.map_coordinates()
            for table in mapping.values():
                display(table)
            display(pipeline.all_rows[['source_file', 'stakes', 'bcpc_arc_length',
                                       'arc_length_parallel', 'arc_length_orthogonal']].head())
            output_dir = pipeline.export(notebook='notebooks/arc_length_layer_fraction_sweep.ipynb')
            figures = pipeline.save_plots()
            for figure in figures.values():
                figure.show()
        finally:
            del pipeline
            gc.collect()
            discard_activation_caches(config.activations_dir)

        run_inference_stage(config, 'Severity', severity_inference, 'severity', model, tokenizer)
        run_inference_stage(config, 'Flipped severity', severity_flipped_inference,
                            'severity_flipped', model, tokenizer)
        run_inference_stage(config, 'Pairwise severity', severity_pairwise_inference,
                            'severity_pairwise', model, tokenizer)
        run_inference_stage(config, 'Severity wording', severity_wording_inference,
                            'severity_wording', model, tokenizer)
        run_inference_stage(config, 'Context variation', context_inference, 'context', model, tokenizer)
    finally:
        del model, tokenizer
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    return output_dir

## Run every layer fraction sequentially

A failed fraction does not prevent later fractions from running. Its traceback is recorded, whatever raw activations it left behind are still discarded, and the cell raises after printing the complete status table. Local model data is cleaned up once, after the last fraction.

In [ ]:
model_name, naming_convention = MODEL
if not isinstance(model_name, str) or not model_name or not isinstance(naming_convention, str) or not naming_convention:
    raise ValueError('MODEL must be a (model_name, naming_convention) pair of nonempty strings.')
naming_convention_spec(naming_convention)
if not LAYER_FRACTIONS:
    raise ValueError('LAYER_FRACTIONS must contain at least one fraction.')
for fraction in LAYER_FRACTIONS:
    if not isinstance(fraction, (int, float)) or isinstance(fraction, bool) or not 0 < fraction < 1:
        raise ValueError(f'Each layer fraction must be a number in (0, 1); got {fraction!r}.')
if len({fraction_tag(fraction) for fraction in LAYER_FRACTIONS}) != len(LAYER_FRACTIONS):
    raise ValueError('LAYER_FRACTIONS contains duplicate fractions.')

NUM_LAYERS = model_layer_count(model_name, naming_convention, HF_CACHE_DIR)
selected_layers = {fraction: cached_layer_for_fraction(model_name, NUM_LAYERS, fraction)
                   for fraction in LAYER_FRACTIONS}
if len(set(selected_layers.values())) != len(selected_layers):
    raise ValueError(f'Distinct fractions resolve to the same layer for {model_name}: {selected_layers}')
print(f'{model_name}: {NUM_LAYERS} layers')
display(selected_layers)

run_results = {}
try:
    for fraction in LAYER_FRACTIONS:
        tag = fraction_tag(fraction)
        try:
            output_dir = process_fraction(model_name, naming_convention, NUM_LAYERS, fraction)
            run_results[tag] = {'status': 'complete', 'fraction': fraction,
                                'layer': selected_layers[fraction], 'output_dir': str(output_dir)}
        except Exception as exc:
            run_results[tag] = {'status': 'failed', 'fraction': fraction,
                                'layer': selected_layers[fraction],
                                'error': f'{type(exc).__name__}: {exc}',
                                'traceback': traceback.format_exc()}
            print(run_results[tag]['traceback'])
        finally:
            try:
                discard_activation_caches(SWEEP_ROOT / tag)
            except Exception as cleanup_error:
                cleanup_message = f'{type(cleanup_error).__name__}: {cleanup_error}'
                run_results.setdefault(tag, {'status': 'failed'})['cleanup_error'] = cleanup_message
                print(f'WARNING: activation cleanup failed for {tag}: {cleanup_message}')
finally:
    try:
        delete_local_model_data(model_name, HF_CACHE_DIR)
    except Exception as cleanup_error:
        cleanup_message = f'{type(cleanup_error).__name__}: {cleanup_error}'
        run_results.setdefault('model_cache', {'status': 'failed'})['cleanup_error'] = cleanup_message
        print(f'WARNING: cleanup failed for {model_name}: {cleanup_message}')

display(run_results)
failures = {name: result for name, result in run_results.items() if result.get('status') != 'complete'}
cleanup_failures = {name: result for name, result in run_results.items() if 'cleanup_error' in result}
if failures or cleanup_failures:
    raise RuntimeError(f'Pipeline failures: {list(failures)}; cleanup failures: {list(cleanup_failures)}')

## Reusing an exported surface

Each fraction exports its own bundle, so point `config` (or the directory directly) at the fraction you want, for example `artifacts/layer_fraction_sweep/frac60/Qwen3-32B/surface`.

```python
from scripts.stakes_surface_bundle import load_surface_bundle, project_saved_bcpc
arrays, metadata, saved_coordinates = load_surface_bundle(config.surface_dir)
bcpc_projection = project_saved_bcpc(activation_batch, arrays, metadata)
pls_scores = (activation_batch - arrays['pls_mean']) @ arrays['pls_rotations']
new_surface_coordinates = saved_coordinates.map_points(pls_scores[:, :3])
```

Because raw activations are discarded, re-projecting a batch requires recomputing its activations with the layer that fraction used.